# N03 — Surrogate zoo (benchmark via `run_benchmark`)

**Purpose**: Run the full surrogate family against the teacher using
`decentra.experiments.run_benchmark`. Covers:

- `Tree-d1`, `Tree-d1-mono`, `Tree-d3`, `Tree-d6`
- `EBM`, `EBM-mono`
- `Ridge`, `OptBin+Ridge`
- `Tree-d1 → Scorecard` (post-conversion)
- `EBM → Scorecard` (post-conversion)
- Oracle (teacher TreeSHAP, degenerate reference)

All surrogates produce `adverse_contributions` with `target_scale='score'`, and
attribution fidelity is computed via name-aware alignment (`metrics.named`).

**Outputs** (`outputs/N03/`):
- `bench_{name}.pkl` + `.json` per dataset.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pickle, numpy as np, pandas as pd
from pathlib import Path
from decentra.surrogate import TreeSurrogate, EBMSurrogate, LinearSurrogate, BinningSurrogate
from decentra.experiments import BenchmarkConfig, run_benchmark

N01 = Path('../outputs/N01'); N02 = Path('../outputs/N02')
OUT = Path('../outputs/N03'); OUT.mkdir(parents=True, exist_ok=True)
with open(N01/'datasets.pkl','rb') as f: datasets = pickle.load(f)
print('Ready')

## 1. Define surrogate factories

Monotone constraints are auto-detected from the training data (Spearman sign of
y_logit vs x_j). This is relevant to credit-scoring audit requirements.

In [ ]:
def make_factories(n_jobs_ebm=8):
    return {
        'Oracle':         lambda fn: TreeSurrogate(max_depth=1, verbose=0),   # overwritten after
        'Tree-d1':        lambda fn: TreeSurrogate(max_depth=1, verbose=0),
        'Tree-d1-mono':   lambda fn: TreeSurrogate(max_depth=1, verbose=0,
                                                    monotone_detect_mode='auto'),
        'Tree-d3':        lambda fn: TreeSurrogate(max_depth=3, verbose=0),
        'Tree-d6':        lambda fn: TreeSurrogate(max_depth=6, verbose=0),
        'EBM':            lambda fn: EBMSurrogate(interactions=0, n_jobs=n_jobs_ebm),
        'EBM-mono':       lambda fn: EBMSurrogate(interactions=0, n_jobs=n_jobs_ebm,
                                                   monotone_detect_mode='auto'),
        'Ridge':          lambda fn: LinearSurrogate(method='ridge', alpha=1.0),
        'OptBin+Ridge':   lambda fn: BinningSurrogate(method='ridge', alpha=1.0,
                                                       max_n_bins=10),
    }

## 2. Run benchmark per dataset

In [ ]:
results = {}
for name, d in datasets.items():
    print(f'\n{"="*60}\n{name}\n{"="*60}')
    shap_te = np.load(N02/f'bb_shap_{name}.npy')
    score_te = np.load(N02/f'bb_score_test_{name}.npy')
    score_tr = np.load(N02/f'bb_score_train_{name}.npy')
    prob_te  = np.load(N02/f'bb_prob_{name}.npy')
    with open(N02/f'train_val_idx_{name}.pkl','rb') as f:
        tv = pickle.load(f)

    config = BenchmarkConfig(
        surrogates=make_factories(),
        reject_percentile=90, target_scale='score',
        ks=(1,3,4), adv_ks=(1,4), missing_policy='zero',
    )
    res = run_benchmark(
        teacher=None,
        X_train=d['X_train'], X_test=d['X_test'],
        y_train_target=score_tr, y_test_binary=d['y_test'].values,
        bb_shap_test=shap_te, bb_prob_test=prob_te, bb_score_test=score_te,
        feature_names=d['feature_names'],
        config=config,
        train_val_split=(tv['train_pos'], tv['val_pos']),
    )
    res.save(OUT/f'bench_{name}', save_models=True)
    df = res.to_dataframe()
    cols = ['surrogate','R2','Spearman','Agree','Top4','AdvTop1','AdvTop4','AdvFull_R','coverage_surr','fit_seconds']
    print(df[cols].round(4).to_string(index=False))
    results[name] = res

print('\nRandom baselines:')
for n,r in results.items():
    print(f'  {n}: {r.info["random_baseline"]}')

## 3. Scorecard conversion (Tree-d1, EBM)

Apply `to_scorecard_model()` to the fitted depth-1 tree and EBM, then re-compute
fidelity on the scorecard output. This documents the fidelity loss (if any)
incurred by the score-book format.

In [ ]:
from sklearn.metrics import r2_score
from decentra.metrics.named import attribution_fidelity_named

sc_rows = []
for name, res in results.items():
    print(f'\n--- {name} scorecard conversion ---')
    X_te = datasets[name]['X_test']
    shap_te = np.load(N02/f'bb_shap_{name}.npy')
    score_te = np.load(N02/f'bb_score_test_{name}.npy')
    prob_te = np.load(N02/f'bb_prob_{name}.npy')
    reject = prob_te >= np.percentile(prob_te, 90)
    bb_adv = pd.DataFrame(shap_te, columns=datasets[name]['feature_names'])
    for base in ['Tree-d1', 'Tree-d1-mono', 'EBM']:
        if base not in res.models: continue
        try:
            sc = res.models[base].to_scorecard_model(
                datasets[name]['X_train'], y_binary=datasets[name]['y_train'],
                feature_names=datasets[name]['feature_names'],
                max_bins_per_feature=5, min_bin_ratio=0.05,
            )
        except Exception as e:
            print(f'  {base}: scorecard conversion failed ({e})'); continue
        pred = sc.predict(X_te)
        contribs_df = pd.DataFrame(sc.contributions(X_te), columns=datasets[name]['feature_names'])
        adv_df = -contribs_df  # score scale
        r2 = r2_score(score_te, pred)
        fid = attribution_fidelity_named(bb_adv, adv_df, reject, ks=(1,3,4), adv_ks=(1,4))
        row = {'dataset': name, 'surrogate': f'{base}→Scorecard', 'R2': r2, **fid}
        sc_rows.append(row)
        print(f'  {base}→Scorecard  R²={r2:.4f}  AT-1={fid["AdvTop1"]:.3f}  AT-4={fid["AdvTop4"]:.3f}')

import json
with open(OUT/'scorecard_fidelity.json','w') as f:
    json.dump(sc_rows, f, indent=2, default=float)